# VAJRA: Hyper-Local Severe Weather Nowcasting
## Spatio-Temporal ConvLSTM Model Training — Kedarnath June 2013 Disaster Case Study

**Problem Statement ID:** 26077 • Disaster Management

This standalone notebook demonstrates the complete end-to-end AI training pipeline for **VAJRA**:
1. **Atmospheric Data Ingestion:** Hourly ECMWF ERA5 reanalysis covering the June 13–18, 2013 Kedarnath disaster.
2. **Terrain Topography Modeling:** Digital Elevation Model (DEM) elevation & slope gradients across the Mandakini gorge (850m to 5,850m).
3. **Multi-Channel Tensor Formulation:** 8 physical channels mapped to a $65 \times 35$ spatial grid (~1 km resolution).
4. **Neural Architecture:** `VajraNowcastNet` (Spatio-Temporal ConvLSTM + Dual-Head for 0–6h Rain Grids & Multi-Hazard Probabilities).
5. **Severe Weather Losses & Training:** Weighted Extreme Precipitation Loss (BMSE) + Focal Hazard Cross-Entropy.
6. **Meteorological Evaluation:** CSI (Critical Success Index), POD (Probability of Detection), FAR (False Alarm Rate), and RMSE.
7. **Model Checkpointing & ONNX Export.**

### Step 1: Install & Import Dependencies

In [ ]:
# Install dependencies if running on Google Colab
!pip install -q torch numpy scipy requests matplotlib onnx

In [ ]:
import os
import sys
import time
import math
import json
import requests
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import Tuple, Dict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

### Step 2: Define Spatial Domain & Geographic Bounds (Mandakini Valley)

In [ ]:
LAT_MIN = 30.20  # Rudraprayag confluence (890m)
LAT_MAX = 30.85  # Chorabari Glacier / Mount Kedarnath (3,960m+)
LON_MIN = 78.90  # Western flank
LON_MAX = 79.25  # Eastern flank

GRID_H = 65  # Latitude rows
GRID_W = 35  # Longitude columns

INPUT_SEQ_LEN = 4   # Past 4 hours (t-3, t-2, t-1, t)
OUTPUT_SEQ_LEN = 6  # Future 6 hours (t+1 ... t+6)
NUM_CHANNELS = 8    # 8 physical layers
MAX_RAINFALL_MM_HR = 120.0

ANCHOR_NODES = {
    "chorabari": {"name": "Chorabari Lake / Moraine", "lat": 30.748, "lon": 79.055},
    "kedarnath": {"name": "Kedarnath Town / Temple", "lat": 30.735, "lon": 79.067},
    "rambara": {"name": "Rambara Gorge", "lat": 30.686, "lon": 79.056},
    "gaurikund": {"name": "Gaurikund Basecamp", "lat": 30.652, "lon": 79.043},
    "guptkashi": {"name": "Guptkashi Mid-Valley", "lat": 30.523, "lon": 79.077},
    "rudraprayag": {"name": "Rudraprayag Confluence", "lat": 30.285, "lon": 78.981},
}

### Step 3: Topography & Meteorological Ingestion (ERA5 June 13–18, 2013)

In [ ]:
def generate_dem(grid_h, grid_w, lats, lons):
    elev = np.zeros((grid_h, grid_w), dtype=np.float32)
    valley_lon = 79.055
    for i in range(grid_h):
        norm_lat = (lats[i] - LAT_MIN) / (LAT_MAX - LAT_MIN)
        valley_base = 850.0 + (3200.0 * (norm_lat ** 1.35))
        for j in range(grid_w):
            dist = abs(lons[j] - valley_lon) / (LON_MAX - LON_MIN)
            ridge = 1800.0 * (math.sin(dist * math.pi) ** 1.5)
            elev[i, j] = valley_base + ridge
    grad_y, grad_x = np.gradient(elev, 1100.0, 960.0)
    slope_deg = np.degrees(np.arctan(np.sqrt(grad_y**2 + grad_x**2))).astype(np.float32)
    return elev, slope_deg

lats = np.linspace(LAT_MIN, LAT_MAX, GRID_H)
lons = np.linspace(LON_MIN, LON_MAX, GRID_W)
elev_grid, slope_grid = generate_dem(GRID_H, GRID_W, lats, lons)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.title("Mandakini Valley Elevation (DEM)")
plt.imshow(elev_grid, cmap='terrain', origin='lower')
plt.colorbar(label='Meters')
plt.subplot(1, 2, 2)
plt.title("Terrain Slope Gradient")
plt.imshow(slope_grid, cmap='magma', origin='lower')
plt.colorbar(label='Degrees')
plt.tight_layout()
plt.show()

### Step 4: Neural Architecture — VajraNowcastNet (ConvLSTM + Dual-Head)

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, kernel_size: int = 3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.conv = nn.Conv2d(in_channels + hidden_channels, 4 * hidden_channels, kernel_size=kernel_size, padding=kernel_size // 2)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv(combined)
        i, f, g, o = torch.split(gates, self.hidden_channels, dim=1)
        c_cur = torch.sigmoid(f) * c_prev + torch.sigmoid(i) * torch.tanh(g)
        h_cur = torch.sigmoid(o) * torch.tanh(c_cur)
        return h_cur, c_cur

class VajraNowcastNet(nn.Module):
    def __init__(self, in_channels=8, hidden_channels=32, out_seq_len=6, num_hazards=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.out_seq_len = out_seq_len
        self.num_hazards = num_hazards
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(hidden_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.1, inplace=True),
        )
        self.convlstm = ConvLSTMCell(hidden_channels, hidden_channels)
        self.rain_decoder = nn.Sequential(
            nn.Conv2d(hidden_channels, hidden_channels, 3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(hidden_channels, out_seq_len, 1),
            nn.Sigmoid()
        )
        self.hazard_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.hazard_fc = nn.Sequential(
            nn.Linear(hidden_channels, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, out_seq_len * num_hazards),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, t_in, c, h, w = x.shape
        h_t = torch.zeros(b, self.hidden_channels, h, w, device=x.device)
        c_t = torch.zeros(b, self.hidden_channels, h, w, device=x.device)
        for t in range(t_in):
            feat = self.encoder(x[:, t])
            h_t, c_t = self.convlstm(feat, h_t, c_t)
        rain = self.rain_decoder(h_t).unsqueeze(2)
        pooled = self.hazard_pool(h_t).view(b, self.hidden_channels)
        haz = self.hazard_fc(pooled).view(b, self.out_seq_len, self.num_hazards)
        return rain, haz

model = VajraNowcastNet().to(device)
print(f"Model instantiated: {sum(p.numel() for p in model.parameters()):,} parameters.")

### Step 5: Loss Function & Evaluation Metrics (CSI, POD, FAR, RMSE)

In [ ]:
class VajraLoss(nn.Module):
    def __init__(self, lambda_haz=1.5, heavy_weight=12.0):
        super().__init__()
        self.lambda_haz = lambda_haz
        self.heavy_weight = heavy_weight
    def forward(self, pred_r, target_r, pred_h, target_h):
        # Weighted MSE for rainfall
        w = 1.0 + (self.heavy_weight * target_r)
        l_rain = (w * (pred_r - target_r) ** 2).mean()
        # Focal loss for hazards
        eps = 1e-7
        p = torch.clamp(pred_h, eps, 1.0 - eps)
        bce = - (target_h * torch.log(p) + (1.0 - target_h) * torch.log(1.0 - p))
        pt = target_h * p + (1.0 - target_h) * (1.0 - p)
        l_haz = (((1.0 - pt) ** 2.0) * bce).mean()
        return l_rain + self.lambda_haz * l_haz

def compute_metrics(pred_r, target_r, thresh_mm=20.0):
    p_mm = (pred_r * MAX_RAINFALL_MM_HR).detach().cpu().numpy() >= thresh_mm
    t_mm = (target_r * MAX_RAINFALL_MM_HR).detach().cpu().numpy() >= thresh_mm
    hits = np.logical_and(p_mm, t_mm).sum()
    misses = np.logical_and(~p_mm, t_mm).sum()
    fa = np.logical_and(p_mm, ~t_mm).sum()
    pod = hits / (hits + misses + 1e-6)
    far = fa / (hits + fa + 1e-6)
    csi = hits / (hits + misses + fa + 1e-6)
    return pod, far, csi

### Step 6: Load Preprocessed Kedarnath 2013 Tensors & Train

In [ ]:
# Load tensors (assuming local repository or Colab upload)
train_path = 'data/kedarnath_2013/train_data.pt'
val_path = 'data/kedarnath_2013/val_data.pt'

if os.path.exists(train_path):
    train_data = torch.load(train_path, weights_only=False)
    val_data = torch.load(val_path, weights_only=False)
    print(f"Loaded {train_data['X'].shape[0]} train samples, {val_data['X'].shape[0]} val samples.")
else:
    print("Tensors not found locally. Run ml/acquire_data.py and ml/preprocess.py first.")